# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmerSajid842/flyrankmlproject/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> This notebook uses the local starter dataset (`content_refresh_anonymized.csv`) instead of the Hugging Face warehouse dataset to complete the data contract exercise.

## 1. Contract (Markdown)

### What does one row mean?

One row represents one content page with its aggregated search performance metrics over the last 90 days.

---

### Which dataset am I using?

Dataset:
data/raw/content_refresh_anonymized.csv

> This notebook uses the local starter dataset (`content_refresh_anonymized.csv`) instead of the Hugging Face warehouse dataset to complete the data contract exercise.

---

### Time Window

The dataset contains aggregated metrics for the previous 90 days along with last 30-day and previous 30-day statistics.

---

### What am I predicting?

I want to rank content pages according to their refresh priority.

---

### What am I deliberately excluding?

I will not use future performance information or manually created labels because they would introduce data leakage.

# 2. Verification Queries (Code)

### Query 1

In [15]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path.cwd() / "data/raw/content_refresh_anonymized.csv",
    Path.cwd().parent / "data/raw/content_refresh_anonymized.csv",
    Path.cwd().parent.parent / "data/raw/content_refresh_anonymized.csv",
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Starter data not found in the expected repo locations.")

df = pd.read_csv(data_path)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


Rows: 30000
Columns: 44


The dataset contains 30,000 rows and 44 columns.

### Query 2

In [17]:
print(df["content_id"].nunique())

30000


Each row represents one content page.

### Query 3

In [16]:
print(df.isnull().sum())

content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc                        2468
content_type                  0
main_intent                2374
word_count                 7699
char_count                 7699
provider_used             21438
model_used                 5733
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
sessions_last_30d             0
impressions_prev_30d          0
clicks_prev_30d               0
sessions_prev_30d             0
content_age_days              0
age_tier                      0
age_tier_order                0
days_sin

Some columns such as search_volume, competition and word_count contain missing values.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df.columns.tolist())
df.info()

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_i

## 3. Feature Frame

Create only FIVE features.

In [10]:
features = df[
[
"impressions_90d",
"clicks_90d",
"ctr",
"avg_position",
"days_since_last_update"
]]

display(features.head())

,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update
0,3803,29,0.76,10.6,20
1,15320,7,0.05,20.3,25
2,12581,11,0.09,36.5,20
3,11751,58,0.49,6.2,22
4,19140,24,0.13,44.0,14


## 4. Leakage Experiment

Create a deliberately bad feature.

In [11]:
df["bad_feature"] = df["trend_direction"]
display(df[["trend_direction", "bad_feature"]].head())

,trend_direction,bad_feature
0,down,down
1,down,down
2,down,down
3,stable,stable
4,down,down


This feature contains information very close to the prediction target. Using it would give the model unfair future knowledge, producing unrealistically high accuracy. Therefore it was removed.

In [12]:
df.drop(columns=["bad_feature"], inplace=True)

### impressions_90d
`impressions_90d` is a historical metric reflecting past search visibility, thus it's available before making any predictions.

### clicks_90d
`clicks_90d` summarizes historical clicks, making it available as a feature before prediction.

### ctr
`ctr` (Click-Through Rate) is calculated from historical clicks and impressions, so it is already known before making a prediction.

### avg_position
`avg_position` comes directly from historical Google Search performance data, so it is available before prediction.

### days_since_last_update
`days_since_last_update` is known because the page's last update date is a pre-existing fact, available before prediction.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
## Limitations

This dataset is anonymized and contains aggregated historical metrics. It does not include actual page content or future observations, so conclusions are limited to decision support rather than proving causal effects.
*   Dataset is anonymized.
*   Missing values exist in several columns.
*   The dataset cannot explain why traffic changes.
*   Results may not generalize to every website.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

✔ One row defined

✔ Dataset defined

✔ Five features selected

✔ Leakage demonstrated

✔ Limitation explained